# 07 - Declarative pipelines, expectations, and serving objects

Prerequisites / execution context: Run 00 and 01. Notebook is an instructional controller; pipeline definition files must be added as pipeline source, not executed as ordinary interactive notebooks.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R32: Lakeflow pipelines](https://docs.databricks.com/aws/en/ldp/); [R33: Develop pipelines in Python](https://docs.databricks.com/aws/en/ldp/developer/python-dev); [R34: Pipeline expectations](https://docs.databricks.com/aws/en/ldp/expectations); [R35: Materialized views](https://docs.databricks.com/aws/en/ldp/concepts/materialized-views); [R36: Streaming tables](https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Configure the pipeline
Create an ETL pipeline in Jobs & Pipelines. Select serverless if available, a destination catalog, and a NEW schema such as dea_riccardo_pipeline. Add pipelines/orders_pipeline.py from this kit as source. Add configuration setting kit.bronze = workspace.dea_riccardo_bronze (adapt your catalog/prefix). Use triggered mode and run an update. The code deliberately uses unique pipeline-owned outputs.


## 2. Read the source before running
@dp.table declares a streaming table when its function returns a streaming DataFrame. @dp.materialized_view declares a materialized view over a batch DataFrame. Definition functions return DataFrames and avoid side effects. Expectations in this file drop invalid amounts and report metrics. Official Databricks documentation: [R33: Develop pipelines in Python](https://docs.databricks.com/aws/en/ldp/developer/python-dev); [R34: Pipeline expectations](https://docs.databricks.com/aws/en/ldp/expectations).


In [ ]:
PIPELINE_SCHEMA=f"{CATALOG}.{PREFIX}_pipeline"
# Execute only AFTER the pipeline update succeeds.
CHECK_PIPELINE=False
if CHECK_PIPELINE:
    assert spark.table(f"{PIPELINE_SCHEMA}.orders_valid").count()==6
    assert spark.table(f"{PIPELINE_SCHEMA}.order_metrics").first().valid_event_rows==6


## 3. Observe and explain
Inspect the pipeline graph, update details, and expectation metrics. Expected 8 incoming raw events, 2 dropped by the positive_amount rule, 6 valid events. The materialized view reports valid EVENT rows, not deduplicated orders: amount total 420.00. This differs intentionally from the latest-order Gold revenue of 220.00 in lab 02. Do not confuse event and order grains.

Re-run an update without changing input and inspect the results. Then append one valid new event to the Bronze table and run another update. For this exercise do not overwrite or update the append-only source table while the streaming pipeline is in use. Official Databricks documentation: [R36: Streaming tables](https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables).


## 4. Change the quality policy
On a new disposable pipeline, replace expect_or_drop with expect to retain violations and collect metrics; then with expect_or_fail to reject the update. Compare behavior and describe which business requirement each satisfies. Quarantine is a separate routing design, not an automatic property of drop. Official Databricks documentation: [R34: Pipeline expectations](https://docs.databricks.com/aws/en/ldp/expectations).


## Serving-object decision exercise
Use a regular view for lightweight reusable query logic, a managed table for explicit write control, a streaming table for incremental ingestion, and a materialized view for a managed precomputed query result. A materialized view is correct at refresh time; do not promise every refresh is incremental. Official Databricks documentation: [R35: Materialized views](https://docs.databricks.com/aws/en/ldp/concepts/materialized-views); [R36: Streaming tables](https://docs.databricks.com/aws/en/ldp/concepts/streaming-tables).
